**Import Libraly**

In [52]:
import pandas as pd
import numpy as np
import re
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Ridge
import warnings
warnings.filterwarnings("ignore")


**Load Data**

In [53]:
df = pd.read_csv("E:/Project/FootballerPlayerStatistical/data/results_min900_with_value.csv", encoding = "utf-8-sig")
print(df.shape)
print(df.head())
print("Missing Estimated Value:", df["Estimated Value"].isna().sum())

(300, 60)
               Player             Team   Nation Position   Age    MP  \
0      Aaron Ramsdale      Southampton  eng ENG       GK  26.0   N/a   
1   Aaron Wan-Bissaka  West Ham United   cd COD    DF,MF  26.0  36.0   
2  Abdoulaye Doucouré          Everton   ml MLI       MF  31.0   N/a   
3      Adam Armstrong      Southampton  eng ENG    FW,MF  27.0   N/a   
4          Adam Smith      Bournemouth  eng ENG       DF  33.0   N/a   

   Playing Time_Starts  Playing Time_Min  Playing Time_90s  Performance_Gls  \
0                   30            2700.0              30.0              0.0   
1                   35            3154.0              35.0              2.0   
2                   31            2564.0              28.5              3.0   
3                   15            1248.0              13.9              2.0   
4                   19            1590.0              17.7              0.0   

   ...  Performance_Fls  Performance_Fld  Performance_Off  Performance_Crs  \
0  .

**Chuyển Estimated Value -> số**

In [54]:
def parse_ft_value(x):
    if not isinstance(x, str):
        return np.nan
    s = x.strip().replace(",", ".")
    s = s.replace("€", "").replace("EUR", "").strip()
    m = re.search(r"([0-9]+(?:\.[0-9]+)?)\s*(bn|b|m|k)\b", s, re.I)
    if not m:
        return np.nan
    val = float(m.group(1))
    suf = m.group(2).lower()
    if suf in ("bn","b"): val *= 1000.0
    elif suf == "k":     val /= 1000.0
    return val  # million EUR

df["Value_EUR_m"] = df["Estimated Value"].apply(parse_ft_value)
print("Missing:", df["Value_EUR_m"].isna().sum())

Missing: 89


In [55]:
print(df["Value_EUR_m"].describe())
print("Missing Value_EUR_m:", df["Value_EUR_m"].isna().sum())

count    211.000000
mean      35.945024
std       25.893434
min        1.000000
25%       16.700000
50%       30.200000
75%       49.300000
max      145.500000
Name: Value_EUR_m, dtype: float64
Missing Value_EUR_m: 89


**Tạo Dataset để train (loại bỏ các cột có giá trị missing value)**

In [56]:
df_model = df.dropna(subset=["Value_EUR_m"]).copy()
# Clean N/a values ngay từ đầu
df_model = df_model.replace(["N/a", "N/A", "NA", ""], np.nan)
df_model = df_model.infer_objects(copy=False)
print("Rows for training:", df_model.shape[0])

Rows for training: 211


**Chọn các cột đặc trưng và biến mục tiêu**

In [57]:
target_col = "Value_EUR_m"

drop_cols = ["Player", "Nation", "Estimated Value"]
drop_cols = [c for c in drop_cols if c in df_model.columns]

feature_cols = [c for c in df_model.columns if c not in drop_cols + [target_col]]

X = df_model[feature_cols].copy()
y = df_model[target_col].copy()

print("X.shape:", X.shape)
print("y.shape:", y.shape)

X.shape: (211, 57)
y.shape: (211,)


**Tách cột số và cột phân loại**

In [44]:
cat_cols = [c for c in ["Position", "Team"] if c in X.columns]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Categorical columns:", len(cat_cols))
print("Numerical columns:", len(num_cols))

Categorical columns: 2
Numerical columns: 55


**Preprocessing Pipeline**

In [45]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),   # Ridge/Linear cần
        ]), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols),
    ]
)

#log-transform target
y_log = np.log1p(y)

**Chia tập train/test**

In [46]:
X_train, X_test, y_train, y_test = train_test_split(X, y_log, test_size=0.2, random_state=42)
print("X_train.shape:", X_train.shape)
print("y_train.shape:", y_train.shape)

X_train.shape: (168, 57)
y_train.shape: (168,)


**Model Ridge**

In [47]:
model = Pipeline(steps=[
    ("prep", preprocess),
    ("reg", Ridge(alpha=1.3, random_state=42))
])

print("Training model...")
model.fit(X_train, y_train)

Training model...


,steps,"[('prep', ...), ('reg', ...)]"
,transform_input,None
,memory,None
,verbose,False
,transformers,"[('num', ...), ('cat', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True


**Đánh giá Model**

In [48]:
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Predictions
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

# Calculate metrics
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)
train_rmse = np.sqrt(mean_squared_error(y_train, y_train_pred))
test_rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
train_mae = mean_absolute_error(y_train, y_train_pred)
test_mae = mean_absolute_error(y_test, y_test_pred)

print("=== Model Performance ===")
print(f"Train R² Score: {train_r2:.4f}")
print(f"Test R² Score:  {test_r2:.4f}")
print(f"\nTrain RMSE: {train_rmse:.4f}")
print(f"Test RMSE:  {test_rmse:.4f}")
print(f"\nTrain MAE: {train_mae:.4f}")
print(f"Test MAE:  {test_mae:.4f}")

=== Model Performance ===
Train R² Score: 0.8490
Test R² Score:  0.7597

Train RMSE: 0.3199
Test RMSE:  0.4176

Train MAE: 0.2407
Test MAE:  0.2920


**GridSearch chọn Alpha tốt nhất**

In [58]:
from sklearn.model_selection import GridSearchCV

ridge_pipe = Pipeline(steps=[
    ("prep", preprocess),
    ("reg", Ridge())
])


param_grid = {
    "reg__alpha": np.logspace(-3, 3, 25)
}

gs = GridSearchCV(
    ridge_pipe,
    param_grid=param_grid,
    cv=5,
    scoring="r2",
    n_jobs=-1
)

gs.fit(X_train, y_train)

print("Best alpha:", gs.best_params_["reg__alpha"])
print("CV best R2:", gs.best_score_)

best_model = gs.best_estimator_

Best alpha: 1.7782794100389228
CV best R2: 0.6372506334791614


In [59]:
def evaluate_model(model, X_test, y_test_log, name="model"):
    y_pred_log = model.predict(X_test)
    y_true = np.expm1(y_test_log)
    y_pred = np.expm1(y_pred_log)

    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = mean_squared_error(y_true, y_pred) ** 0.5

    return {
        "model": name,
        "R2": r2,
        "MAE_million_EUR": mae,
        "RMSE_million_EUR": rmse
    }

**So sánh**

In [61]:
results = []
results.append(evaluate_model(model, X_test, y_test, name="Ridge alpha=1 (baseline)"))
results.append(evaluate_model(best_model, X_test, y_test, name="Ridge best alpha (CV)"))

compare_df = pd.DataFrame(results)
print(compare_df)

                      model        R2  MAE_million_EUR  RMSE_million_EUR
0  Ridge alpha=1 (baseline)  0.743019        10.736899         15.855546
1     Ridge best alpha (CV)  0.735782        10.683732         16.077245
